In [ ]:
import sys
from pathlib import Path

repo_root = Path(".").resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import h5py
import random

# Feature normalization explorer

Scans raw charge values from the DUNE APA sparse dataset (default DINO training config)
to help pick `min_val` / `max_val` for a `LogTransform`.

In [ ]:
# --- mirrors DINOConfig defaults ---
DATADIR    = "/nfs/data/1/yuhw/cffm-data/prod-jay-100k-truth-2026-02-27"
CACHE_DIR  = "../data"
APA        = 0
VIEW       = "W"
FRAME_NAME = "frame_rebinned_reco"

In [ ]:
from loader.apa_sparse_meta_dataset import APASparseMetaDataset, CLASS_NAMES

dataset = APASparseMetaDataset(
    datadir=DATADIR,
    apa=APA,
    view=VIEW,
    use_cache=True,
    cache_dir=CACHE_DIR,
)
print(f"Dataset: APA={APA}, view={VIEW} — {len(dataset):,} samples found")
print(f"Classes: {CLASS_NAMES}")

In [ ]:
n_samples = 10000  # adjust as needed (100–5000)

In [ ]:
random.seed(42)
n = min(n_samples, len(dataset))
indices = random.sample(range(len(dataset)), n)

all_feats_list  = []
all_labels_list = []  # one label per pixel, inherited from its event

for i in indices:
    s = dataset.samples[i]
    with h5py.File(s.path, "r") as f:
        frame  = f[s.group][FRAME_NAME]
        coords = frame["coords"][()]
        feats  = frame["features"][()]
    mask         = (coords[:, 0] >= dataset.ch_start) & (coords[:, 0] < dataset.ch_end)
    event_feats  = feats[mask]
    event_label  = dataset._read_event_truth(s.path, s.group)["label"]  # numuCC=0, nueCC=1, NC=2, -1=unknown
    all_feats_list.append(event_feats)
    all_labels_list.append(np.full(len(event_feats), event_label, dtype=np.int32))

all_feats  = np.concatenate(all_feats_list)
all_labels = np.concatenate(all_labels_list)
print(f"Collected {len(all_feats):,} active pixels from {n} events")
for idx, name in enumerate(CLASS_NAMES):
    cnt = (all_labels == idx).sum()
    print(f"  {name}: {cnt:,} px ({cnt/len(all_labels)*100:.1f}%)")
unknown = (all_labels == -1).sum()
if unknown:
    print(f"  unknown: {unknown:,} px ({unknown/len(all_labels)*100:.1f}%)")

In [ ]:
pcts     = [0, 0.1, 1, 5, 25, 50, 75, 95, 99, 99.9, 99.99, 99.999, 100]
pct_vals = np.percentile(all_feats, pcts)

print("## Raw feature statistics\n")
print(f"{'Percentile':>12} | {'Value':>10}")
print("-" * 27)
for p, v in zip(pcts, pct_vals):
    print(f"{str(p)+'%':>12} | {v:>10.3f}")

print(f"\nMean: {all_feats.mean():.3f}   Std: {all_feats.std():.3f}   "
      f"Negative values: {(all_feats < 0).mean() * 100:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), dpi=200)

ax = axes[0]
ax.hist(all_feats, bins=300, histtype="step", lw=2, alpha=0.85)
ax.set_xlabel("Raw charge value [ADC]")
ax.set_ylabel("Count")
ax.set_title("Semi-log")
ax.set_yscale("log")
ax.grid(alpha=0.5)

ax2 = axes[1]
pos = all_feats[all_feats > 0]
log_bins = np.logspace(np.log10(pos.min()), np.log10(pos.max()), 300)
ax2.hist(pos, bins=log_bins, histtype="step", lw=2, alpha=0.85)
ax2.set_xscale("log")
ax2.set_yscale("log")
ax2.set_xlabel("Raw charge value (log scale) [ADC]")
ax2.set_ylabel("Count")
ax2.set_title("Log-log")
ax2.grid(alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
CLASS_COLORS = ["steelblue", "darkorange", "forestgreen", "gray"]
all_class_names = CLASS_NAMES + ["unknown"]
all_class_indices = list(range(len(CLASS_NAMES))) + [-1]

pos_all = all_feats[all_feats > 0]
log_bins = np.logspace(np.log10(pos_all.min()), np.log10(pos_all.max()), 200)
lin_bins = np.linspace(pos_all.min(), pos_all.max(), 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=200)

for cls_idx, cls_name, color in zip(all_class_indices, all_class_names, CLASS_COLORS):
    mask      = (all_labels == cls_idx)
    feats_cls = all_feats[mask]
    if len(feats_cls) == 0:
        continue
    n_px = mask.sum()
    label_str = f"{cls_name}  ({n_px:,} px)"

    # semi-log: linear x, log y
    axes[0].hist(feats_cls[feats_cls > 0], bins=lin_bins, density=True,
                 histtype="step", color=color, linewidth=1.6, label=label_str)
    #axes[0].axvline(x=10, color="red", linestyle="--", alpha=0.7)
    #axes[0].axvline(x=80000, color="red", linestyle="--", alpha=0.7)


    # log-log
    axes[1].hist(feats_cls[feats_cls > 0], bins=log_bins, density=True,
                 histtype="step", color=color, linewidth=1.6, label=label_str)

for ax, xlog, title in zip(
    axes,
    [False, True],
    ["By-class distribution (semi-log)", "By-class distribution (log-log)"],
):
    ax.set_yscale("log")
    if xlog:
        ax.set_xscale("log")
    ax.set_xlabel("Raw charge value [ADC]")
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## LogTransform parameter tuning

The transform maps charge values to **[-1, +1]** via:

```
y = 2 * (log10(x + min_val) - log10(min_val)) / (log10(max_val + min_val) - log10(min_val)) - 1
```

Adjust `min_val` (noise floor) and `max_val` (saturation point) and watch
how much of the distribution lands outside [-1, +1].

In [ ]:
# Seeded from 1st and 99.9th percentile of the loaded data; adjust as needed
min_val = float(np.percentile(all_feats, 2))
max_val = float(np.percentile(all_feats, 99.999))

print(f"min_val = {min_val:.4f}")
print(f"max_val = {max_val:.1f}")

In [ ]:
def log_transform(x, mn, mx):
    y0 = np.log10(mn)
    y1 = np.log10(mx + mn)
    return 2 * (np.log10(np.clip(x, mn, None) + mn) - y0) / (y1 - y0) - 1

transformed = log_transform(all_feats, min_val, max_val)

frac_below = (transformed < -1.0).mean() * 100
frac_above = (transformed >  1.0).mean() * 100
frac_in    = 100.0 - frac_below - frac_above

fig2, ax = plt.subplots(figsize=(9, 4))
ax.hist(transformed, bins=300, color="forestgreen", alpha=0.85)
ax.axvline(-1.0, color="red",  linestyle="--", linewidth=1.5,
            label=f"\u22121  (min_val = {min_val:.4f})")
ax.axvline( 1.0, color="navy", linestyle="--", linewidth=1.5,
            label=f"+1  (max_val = {max_val:.1f})")
ax.set_xlabel("Transformed value")
ax.set_ylabel("Count")
ax.set_title("LogTransform preview")
ax.set_yscale("log")
ax.legend()
plt.tight_layout()
plt.show()

print(f"In [-1, +1]: {frac_in:.2f}%   Below \u22121: {frac_below:.4f}%   Above +1: {frac_above:.4f}%")

In [ ]:
# Two-panel: transform curve (top) + data distribution (bottom), shared log x-axis
fig, (ax_curve, ax_hist) = plt.subplots(
    2, 1, figsize=(9, 6),
    gridspec_kw={"height_ratios": [3, 1]},
    sharex=True, dpi=200
)

x_plot = np.logspace(np.log10(0.1), np.log10(max_val * 2), 1000)
y_plot = log_transform(x_plot, min_val, max_val)

ax_curve.plot(x_plot, y_plot, color="black", linewidth=2)
ax_curve.axhline(-1, color="red",  linestyle="--", linewidth=1,
                 label="y = −1  (x = 0)")
ax_curve.axhline( 0, color="gray", linestyle=":",  linewidth=1)
ax_curve.axhline(+1, color="navy", linestyle="--", linewidth=1,
                 label=f"y = +1  (max_val = {max_val:.0f} ADC)")
ax_curve.axvline(min_val, color="darkorange", linestyle="--", linewidth=1.4,
                 label=f"min_val = {min_val:.2f} ADC")
ax_curve.axvline(max_val, color="navy",       linestyle="--", linewidth=1.4,
                 label=f"max_val = {max_val:.0f} ADC")
ax_curve.set_ylabel("Transformed value")
ax_curve.set_ylim(-1.3, 1.5)
ax_curve.legend(fontsize=8, loc="upper left")
ax_curve.grid(alpha=0.3)
ax_curve.set_title("LogTransform curve")

pos = all_feats[all_feats > 0]
log_bins = np.logspace(np.log10(pos.min()), np.log10(pos.max()), 200)
ax_hist.hist(pos, bins=log_bins, color="steelblue", alpha=0.8)
ax_hist.set_xscale("log")
ax_hist.set_yscale("log")
ax_hist.set_xlabel("Raw charge value [ADC]")
ax_hist.set_ylabel("Count")
ax_hist.grid(alpha=0.3)
ax_hist.axvline(min_val, color="darkorange", linestyle="--", linewidth=1.4)
ax_hist.axvline(max_val, color="navy",       linestyle="--", linewidth=1.4)

plt.tight_layout()
plt.show()